# Minimaler eigener Runner fuer Task 44

Dieses Notebook ist der Schritt nach dem Human-Agent-Test. Ziel ist nicht LLM-Intelligenz, sondern der Nachweis, dass ein eigener Runner die zwei wichtigen Dateien erzeugen kann:

- `agent_response.json`
- `network.har`

Danach bewertet WebArena-Verified den Run mit `eval-tasks`.

In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
OFFICIAL_REPO

## 1. Voraussetzung: Demo-GitLab laeuft

Falls noch nicht gestartet:

```bash
cd external/webarena-verified
uv run invoke -r examples gitlab-start
```

Der Runner erwartet `http://localhost:8012`.

In [ ]:
subprocess.run(['docker', 'ps', '--filter', 'name=wa-demo-gitlab'], check=True)

## 2. Task-Input fuer Task 44 erzeugen

Der Runner liest `output/tasks.demo.json`. Diese Datei kann jederzeit neu erzeugt werden.

In [ ]:
(OFFICIAL_REPO / 'output').mkdir(exist_ok=True)
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', '44',
    '--config', 'examples/configs/config.demo.json',
    '--output', 'output/tasks.demo.json',
], cwd=OFFICIAL_REPO, check=True)

json.loads((OFFICIAL_REPO / 'output/tasks.demo.json').read_text())

## 3. Eigenen Runner ausfuehren

Der Runner macht absichtlich nur das Minimum fuer Task 44:

1. Task laden.
2. Browser starten.
3. Von `http://localhost:8012` nach `/dashboard/todos` navigieren.
4. HAR speichern.
5. `agent_response.json` mit `NAVIGATE/SUCCESS` schreiben.
6. `eval-tasks` starten.

Mit `tqdm` siehst du die Schritte.

In [ ]:
subprocess.run([
    str(PROJECT_ROOT / '.venv/bin/python'),
    str(PROJECT_ROOT / 'scripts/run_gitlab_task44_navigate_runner.py'),
    '--repo-root', str(OFFICIAL_REPO),
    '--tasks-file', 'output/tasks.demo.json',
    '--task-id', '44',
    '--output-root', 'output/auto-run',
    '--config', 'examples/configs/config.demo.json',
], cwd=PROJECT_ROOT, check=True)

## 4. Artefakte inspizieren

In [ ]:
run_dir = OFFICIAL_REPO / 'output/auto-run/44'
sorted(p.name for p in run_dir.iterdir())

In [ ]:
json.loads((run_dir / 'agent_response.json').read_text())

In [ ]:
eval_result = json.loads((run_dir / 'eval_result.json').read_text())
{key: eval_result.get(key) for key in ['task_id', 'status', 'score']}

## 5. Was bedeutet das fuer den naechsten Schritt?

Wenn dieser Runner `score = 1.0` erreicht, hast du den Human-Agent fuer eine einfache Aufgabe durch eigenen Code ersetzt. Danach kann derselbe Runner schrittweise erweitert werden:

- mehrere Tasks laden
- Schleife ueber Tasks mit `tqdm`
- BrowserGym/AgentLab statt direktem Playwright
- spaeter Planner, Validator, `H`, `k` und Prozessmetriken